<a href="https://colab.research.google.com/github/9terry-student/RetNet/blob/main/RetNet_from_numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

def retention(x):
    # x: (seq_len, d_model)
    seq_len,d_model=x.shape
    # 1. Q, K, V 계산
    # W_Q, W_K, W_V: (d_model, d_model)
    W_q=np.random.randn(d_model,d_model)
    W_k=np.random.randn(d_model,d_model)
    W_v=np.random.randn(d_model,d_model)

    Q=x@W_q
    K=x@W_k
    V=x@W_v

    # 2. 감쇠 행렬 D 만들기
    # D[n][m] = gamma^(n-m) if n >= m else 0
    # gamma = 0.9 (하이퍼파라미터)
    gamma=0.9
    D=np.zeros((seq_len,seq_len))
    for n in range(seq_len):
      for m in range(seq_len):
        if n>=m:
          D[n][m]=gamma**(n-m)

    # 3. Retention 계산
    # output = (Q @ K.T) * D @ V
    output=(Q@K.T)*D@V

    return output

def layernorm(x):
  output=(x-np.mean(x,axis=-1,keepdims=True))/(np.std(x,axis=-1,keepdims=True)+10**(-9))
  return output

def FFN(x):
  d_ff=4*d_model
  w1=np.random.randn(d_model,d_ff)
  w2=np.random.randn(d_ff,d_model)
  b1=np.zeros(d_ff)
  b2=np.zeros(d_model)

  output=np.maximum(0,x@w1+b1)@w2+b2

  return output

def retnet_block(x):
    # 1. LayerNorm → Retention
    output=retention(layernorm(x))

    # 2. Residual
    output+=x

    # 3. LayerNorm → FFN
    # 4. Residual
    output+=FFN(layernorm(output))

    return output


In [2]:
# test
np.random.seed(42)
seq_len = 4
d_model = 16

x = np.random.randn(seq_len, d_model)
output = retnet_block(x)
print("output shape:", output.shape)  # (4, 16)

output shape: (4, 16)
